In [8]:
experiment = "cifar100_segmented"

In [9]:
# To sort the results.csv by instance_id
CSV_PATH = f"../results/{experiment}/results.csv"

import pandas as pd
df = pd.read_csv(CSV_PATH)
df["instance_id"] = pd.to_numeric(df["instance_id"], errors="coerce")
df_sorted = df.sort_values(by="instance_id", ascending=True)
df_sorted.to_csv(CSV_PATH, index=False)

In [11]:
# To Merge the results.csv and instance(vnnlib property) info
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt

results_path = f"../results/{experiment}/results.csv"
stats_path = f"../results/{experiment}/input_change_stats.csv"
out_path = f"../results/{experiment}/combined_results.csv"

df_results = pd.read_csv(results_path)
df_stats   = pd.read_csv(stats_path)

import pandas as pd
import re
import numpy as np

# -------- ImageNet-style (your old one) --------
VN_RE_IMAGENET = re.compile(
    r"(?P<image>n\d+_[^/_]+)_"                         # n01440764_tench
    r"(?:(?P<global>global)|seg(?P<seg>\d+)_(?P<fix>fixmask|fixnonmask))_"
    r"k(?P<k>\d+)_"
    r"eps_(?P<eps>[0-9.]+)"
)

# -------- CIFAR100-style (new) --------
# Example:
# CIFAR100_resnet_large__resnet_large__label_10__idx_7516_global_k1024_eps_0.0039
# CIFAR100_resnet_large__resnet_large__label_10__idx_7516_fix_mask_seg0_k1024_eps_0.0039
VN_RE_CIFAR = re.compile(
    r"^(?P<prefix>CIFAR100_[^_]+__[^_]+__)"            # CIFAR100_resnet_large__resnet_large__
    r"label_(?P<label>\d+)__idx_(?P<idx>\d+)__"        # label_10__idx_7516__
    r"(?:(?P<global>global)|(?P<fix>fix_mask|fix_nonmask)_seg(?P<seg>\d+))_"  # global OR fix_*_segX
    r"k(?P<k>\d+)_"
    r"eps_(?P<eps>[0-9.]+)$"
)

def decode_vnnlib(v: str):
    v = str(v)
    base = v.split("/")[-1].replace(".vnnlib", "")

    # Try CIFAR first (because it uses "__" heavily)
    m = VN_RE_CIFAR.match(base)
    if m:
        k = int(m.group("k"))
        eps = float(m.group("eps"))

        label = int(m.group("label"))
        idx = int(m.group("idx"))

        # build a stable "image" key (choose what you want to merge on)
        # Option A (recommended): use idx as the unique image id
        image = f"idx_{idx}"
        # Option B: include label too:
        # image = f"label_{label}__idx_{idx}"

        if m.group("global") == "global":
            segment_index = -1
            pattern = "global"
        else:
            segment_index = int(m.group("seg"))
            pattern = m.group("fix")  # already fix_mask / fix_nonmask

        return pd.Series({
            "image": image,
            "segment_index": segment_index,
            "pattern": pattern,
            "k": k,
            "eps": eps,
            "label": label,
            "idx": idx
        })

    # Fall back to ImageNet pattern
    m = VN_RE_IMAGENET.search(base)
    if m:
        image = m.group("image")
        k = int(m.group("k"))
        eps = float(m.group("eps"))

        if m.group("global") == "global":
            segment_index = -1
            pattern = "global"
        else:
            segment_index = int(m.group("seg"))
            fix = m.group("fix")
            pattern = {"fixmask": "fix_mask", "fixnonmask": "fix_nonmask"}[fix]

        return pd.Series({
            "image": image,
            "segment_index": segment_index,
            "pattern": pattern,
            "k": k,
            "eps": eps,
            "label": pd.NA,
            "idx": pd.NA
        })

    # If nothing matched
    return pd.Series({
        "image": pd.NA,
        "segment_index": pd.NA,
        "pattern": pd.NA,
        "k": pd.NA,
        "eps": pd.NA,
        "label": pd.NA,
        "idx": pd.NA
    })

# ============================================================
# 2.5) FORCE MERGE-KEY DTYPES (fix object vs float mismatch)
# ============================================================
merge_keys = ["image", "segment_index", "pattern", "k", "eps_key"]

def normalize_keys(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # string keys
    for col in ["image", "pattern"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()
            df.loc[df[col].isin(["<NA>", "nan", "None", ""]), col] = pd.NA

    # integer-ish keys
    for col in ["segment_index", "k"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    # eps_key should be numeric (float)
    if "eps_key" in df.columns:
        df["eps_key"] = pd.to_numeric(df["eps_key"], errors="coerce")

    return df

df_results_decoded = normalize_keys(df_results_decoded)
df_stats = normalize_keys(df_stats)

# (optional) quick debug to confirm
print("Key dtypes (stats):", df_stats[merge_keys].dtypes.to_dict())
print("Key dtypes (results):", df_results_decoded[merge_keys].dtypes.to_dict())

# ============================================================
# 2) Clean types + make merge-safe eps key (avoid float glitches)
# ============================================================
for d in (df_results_decoded, df_stats):
    d["k"] = pd.to_numeric(d["k"], errors="coerce").astype("Int64")
    d["segment_index"] = pd.to_numeric(d["segment_index"], errors="coerce").astype("Int64")
    d["eps"] = pd.to_numeric(d["eps"], errors="coerce")

# robust merge key for eps (string with fixed precision)
EPS_DECIMALS = 10
df_results_decoded["eps_key"] = df_results_decoded["eps"].round(EPS_DECIMALS)
df_stats["eps_key"] = df_stats["eps"].round(EPS_DECIMALS)

# ============================================================
# 3) Merge with input_change_stats on decoded keys
# ============================================================
merge_keys = ["image", "segment_index", "pattern", "k", "eps_key"]

df_merged = df_stats.merge(
    df_results_decoded.drop(columns=["eps"]),  # keep eps from stats (or swap if you prefer)
    on=merge_keys,
    how="left",
    suffixes=("", "_res")
)

# (optional) keep a single eps column
df_merged = df_merged.drop(columns=["eps_key"])

# Save
df_merged.to_csv(out_path, index=False)
print("Saved:", out_path)
print("Merged shape:", df_merged.shape)

# Quick sanity check: rows that didn't find a matching results row
missing = df_merged["result"].isna().sum() if "result" in df_merged.columns else None
print("Unmatched rows (result is NaN):", missing)

KeyError: "['eps_key'] not in index"

In [ ]:
# To Combining the results of two csv files
import pandas as pd

df1 = pd.read_csv("../results/vggnet16_benchmark2022_segmented_all/results.csv")
df2 = pd.read_csv("../results/vggnet16_benchmark2022_segmented_all/results_part1_until_5700.csv")

combined = pd.concat([df1, df2], ignore_index=True)

combined.to_csv("../results/vggnet16_benchmark2022_segmented_all/results_part1_until_5700.csv", index=False)

In [ ]:
# To find the index of an image in a folder
import os

def find_image_index(folder, image_name):
    """
    folder: path to directory containing images
    image_name: base name WITHOUT extension (e.g. 'n02113186_Cardigan')
    """

    # list only files (ignore subdirs)
    files = [
        f for f in os.listdir(folder)
        if os.path.isfile(os.path.join(folder, f))
    ]

    # sort for stable ordering
    files = sorted(files)

    # strip extensions
    base_names = [os.path.splitext(f)[0] for f in files]

    if image_name not in base_names:
        raise ValueError(
            f"'{image_name}' not found in {folder}\n"
            f"Example names: {base_names[:10]}"
        )

    idx = base_names.index(image_name)

    return idx


# ------------------------
# EXAMPLE USAGE
# ------------------------

if __name__ == "__main__":

    FOLDER = "../../benchmarks/vggnet16_benchmark2022/imagenet-sample"
    IMAGE  = "n02113186_Cardigan"

    idx = find_image_index(FOLDER, IMAGE)

    print(f"Image '{IMAGE}' index in folder: {idx-1}")

In [ ]:
# To filter a dataframe to only images that appear exactly 3 times (G O B)
import pandas as pd
import numpy as np

def complete_triplets(df, image_col="image", debug_path=None):
    df = df.copy()
    print("Original shape:", df.shape)

    # keep only images that appear exactly 3 times
    counts = df[image_col].value_counts()
    good_images = counts[counts == 3].index
    df = df[df[image_col].isin(good_images)].copy()

    print("Number of images:", len(good_images))
    print("Shape after filtering to triplets:", df.shape)

    if debug_path is not None:
        df.to_csv(f"{debug_path}_{len(good_images)}_imgs.csv", index=False)


CSV_PATH = f"../../results/{experiment}/combined_results.csv"
OUT_DIR  = f"../results/{experiment}"

df = pd.read_csv(CSV_PATH)

# numeric safety
df["k"] = pd.to_numeric(df["k"], errors="coerce")
df["eps"] = pd.to_numeric(df["eps"], errors="coerce")

# (k, eps) -> subdf
dfs_by_keps = {key: subdf.copy() for key, subdf in df.groupby(["k", "eps"])}

# filter and save each (k, eps)
dfs_by_keps_triplets = {}
for (k, eps), subdf in dfs_by_keps.items():
    print(f"Processing k={k}, eps={eps}")
    dbg = f"{OUT_DIR}/triplets_k{k}_eps{eps}"
    complete_triplets(subdf, debug_path=dbg)

Processing k=10.0, eps=0.0001
Original shape: (843, 24)
Number of images: 278
Shape after filtering to triplets: (834, 24)
Processing k=10.0, eps=0.0003
Original shape: (840, 24)
Number of images: 277
Shape after filtering to triplets: (831, 24)
Processing k=50176.0, eps=0.0001
Original shape: (772, 24)
Number of images: 239
Shape after filtering to triplets: (717, 24)
Processing k=50176.0, eps=0.0003
Original shape: (586, 24)
Number of images: 160
Shape after filtering to triplets: (480, 24)


In [ ]:
# To find BnB Instance (`domains_visited > 0`)
import os
import pandas as pd

experiment = "exp_1/triplets_k50176.0_eps0.0001_239_imgs"

CSV_PATH = f"results/{experiment}.csv"

df = pd.read_csv(CSV_PATH)

assert "domains_visited" in df.columns, "domains_visited column not found!"

df["domains_visited"] = pd.to_numeric(df["domains_visited"], errors="coerce").fillna(0)

bnb_df = df[df["domains_visited"] > 0].copy()

bnb_df = bnb_df.sort_values(
    by="lb_minus_rhs",
    ascending=False
)

print("Rows with domains_visited > 0:", len(bnb_df))
bnb_df.head()


cols = [
    "instance_id",
    "image",
    "tag",
    "is_global",
    "segment_index",
    "eps",
    "k",
    "result",
    "lb_minus_rhs",
    "domains_visited",
    "bab_time",
    "all_time",
]

# Only keep columns that exist (safe if your CSV is slightly different)
cols = [c for c in cols if c in bnb_df.columns]

bnb_df[cols].sort_values("domains_visited", ascending=False)

# OUT_PATH = f"rows_with_bnb_entered_{experiment}.csv"
# os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
# bnb_df.to_csv(OUT_PATH, index=False)
# print("Saved to", OUT_PATH)

In [ ]:
# To merge the results

from pathlib import Path
import csv
# -----------------------------
# Config
# -----------------------------
ROOT = Path("../results")          
OUT  = Path("csvs/all_vggnet16_results_original_6hourstimeout.csv")

rows_out = []
header_out = None

for sub in sorted(ROOT.iterdir()):
    if not sub.is_dir():
        continue
    if not sub.name.lower().startswith("vggnet16_benchmark2022"):
        continue

    csv_path = sub / "results.csv"
    if not csv_path.exists():
        print(f"[WARN] No results.csv in {sub.name}")
        continue

    print(f"[LOAD] {csv_path}")

    with csv_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        # build output header once
        if header_out is None:
            header_out = list(reader.fieldnames) + ["source_folder"]
        else:
            # if columns differ across files, we still handle it safely later
            pass

        for r in reader:
            r["source_folder"] = sub.name
            rows_out.append(r)

if not rows_out:
    raise RuntimeError("No VGGNet16 CSV files found!")

# union of all keys to be safe if some CSVs have extra/missing columns
all_keys = set()
for r in rows_out:
    all_keys.update(r.keys())
all_keys = list(all_keys)

# ensure source_folder is last (nice)
if "source_folder" in all_keys:
    all_keys = [k for k in all_keys if k != "source_folder"] + ["source_folder"]

with OUT.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=all_keys)
    writer.writeheader()
    writer.writerows(rows_out)

print("\nSaved:", OUT)
print("Rows:", len(rows_out))
print("Columns:", len(all_keys))

In [3]:
# To show the table
import csv

CSV = "csvs/all_vggnet16_results_k_all.csv"   # your merged file

with open(CSV, newline="") as f:
    r = csv.reader(f)
    rows = list(r)

# print as a simple aligned table
widths = [max(len(str(x)) for x in col) for col in zip(*rows)]
for i, row in enumerate(rows):
    line = " | ".join(str(x).ljust(w) for x, w in zip(row, widths))
    print(line)
    if i == 0:
        print("-+-".join("-"*w for w in widths))

vnnlib                                                     | timeout | result        | all_time           | onnx              | instance_id | lb_minus_rhs        | bab_time           | domains_visited | init_unstable | source_folder                               
-----------------------------------------------------------+---------+---------------+--------------------+-------------------+-------------+---------------------+--------------------+-----------------+---------------+---------------------------------------------
vnnlib/n02033041_dowitcher_global_k50176_eps_0.0001.vnnlib | 1200    | timeout False | 4005.6531410217285 | onnx/vgg16-7.onnx | 1           | -263.40545654296875 | 3988.4435873031616 | 3               | 0             | vggnet16_benchmark2022_one_img_naive        
vnnlib/n02033041_dowitcher_global_k50176_eps_0.0001.vnnlib | 1200    | timeout False | 4001.934016942978  | onnx/vgg16-7.onnx | 1           | -263.4098205566406  | 3984.5712988376617 | 3               | 0    

In [6]:
# To show the table
import csv

CSV = "csvs/all_vggnet16_results_k_500.csv"   # your merged file

with open(CSV, newline="") as f:
    r = csv.reader(f)
    rows = list(r)

# print as a simple aligned table
widths = [max(len(str(x)) for x in col) for col in zip(*rows)]
for i, row in enumerate(rows):
    line = " | ".join(str(x).ljust(w) for x, w in zip(row, widths))
    print(line)
    if i == 0:
        print("-+-".join("-"*w for w in widths))

vnnlib                                                   | timeout | result      | all_time           | onnx              | instance_id | lb_minus_rhs       | bab_time           | domains_visited | init_unstable | source_folder                                    
---------------------------------------------------------+---------+-------------+--------------------+-------------------+-------------+--------------------+--------------------+-----------------+---------------+--------------------------------------------------
vnnlib/n02033041_dowitcher_global_k500_eps_0.0001.vnnlib | 1200    | unsat False | 842.1632499694824  | onnx/vgg16-7.onnx | 1           | 1.3206205368041992 | 794.1718530654907  | 0               | 0             | vggnet16_benchmark2022_one_img_naive_k500        
vnnlib/n02033041_dowitcher_global_k500_eps_0.0001.vnnlib | 1200    | unsat False | 804.6202256679535  | onnx/vgg16-7.onnx | 1           | 1.3206206560134888 | 793.8658721446991  | 0               | 0         

In [5]:
# To show the table
import csv

CSV = "csvs/all_vggnet16_results_ks_12_14.csv"   # your merged file

with open(CSV, newline="") as f:
    r = csv.reader(f)
    rows = list(r)

# print as a simple aligned table
widths = [max(len(str(x)) for x in col) for col in zip(*rows)]
for i, row in enumerate(rows):
    line = " | ".join(str(x).ljust(w) for x, w in zip(row, widths))
    print(line)
    if i == 0:
        print("-+-".join("-"*w for w in widths))

vnnlib                                                              | domains_visited | init_unstable | result        | timeout | instance_id | lb_minus_rhs        | bab_time           | all_time           | onnx              | source_folder                                  
--------------------------------------------------------------------+-----------------+---------------+---------------+---------+-------------+---------------------+--------------------+--------------------+-------------------+------------------------------------------------
vnnlib/n02033041_dowitcher_global_k50176_eps_0.0001.vnnlib          |                 |               | error         | 1200    | 1           |                     |                    |                    | onnx/vgg16-7.onnx | vggnet16_benchmark2022_one_img_original        
vnnlib/n02033041_dowitcher_seg0_fixmask_k50176_eps_0.0001.vnnlib    | 0               | 0             | unsat False   | 1200    | 2           | 1.3113386631011963  | 885.98